# 03. ReAct 에이전트

## 학습 목표
- ReAct(Reasoning + Acting) 패턴의 개념을 이해한다
- `create_react_agent`로 에이전트를 1줄로 생성할 수 있다
- Tool을 정의하고 에이전트에 연결하여 다중 스텝 추론을 수행할 수 있다
- 시스템 프롬프트로 에이전트에 역할을 부여할 수 있다
- 대화 히스토리를 유지하는 에이전트를 구현할 수 있다

## 1. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env 파일에 설정되어 있는지 확인하세요!"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-mini")

print("환경 설정 완료!")

환경 설정 완료!


## 2. Tool 3개 정의

에이전트가 사용할 도구를 `@tool` 데코레이터로 정의합니다.

In [2]:
from langchain.tools import tool
from datetime import datetime

@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색합니다. 최신 정보나 사실 확인이 필요할 때 사용합니다."""
    # 시뮬레이션 (실제로는 검색 API 호출)
    mock_results = {
        "LangGraph": "LangGraph는 LangChain 팀이 개발한 에이전트 오케스트레이션 프레임워크입니다. 상태 기반 그래프로 복잡한 AI 워크플로우를 구성할 수 있습니다.",
        "ReAct": "ReAct는 Reasoning과 Acting을 결합한 에이전트 패턴입니다. LLM이 생각(Thought) -> 행동(Action) -> 관찰(Observation) 사이클을 반복합니다.",
        "GPT-5": "GPT-5는 OpenAI의 최신 대규모 언어 모델입니다.",
    }
    for key, value in mock_results.items():
        if key.lower() in query.lower():
            return value
    return f"'{query}'에 대한 검색 결과: 관련 정보를 찾았습니다. AI 에이전트 기술이 빠르게 발전하고 있습니다."

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다. Python 수식을 입력하세요."""
    try:
        # 안전한 수식 평가 (기본 연산만 허용)
        allowed_names = {"abs": abs, "round": round, "min": min, "max": max, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    now = datetime.now()
    return now.strftime("%Y년 %m월 %d일 %H시 %M분")

tools = [search_web, calculate, get_current_time]
print(f"정의된 Tool 목록: {[t.name for t in tools]}")

정의된 Tool 목록: ['search_web', 'calculate', 'get_current_time']


## 3. create_react_agent로 에이전트 생성 (1줄!)

`create_react_agent`는 LLM + Tool을 받아서 완전한 ReAct 에이전트 그래프를 자동으로 만들어줍니다.

In [3]:
from langgraph.prebuilt import create_react_agent

# 에이전트 생성 - 이것이 전부입니다!
agent = create_react_agent(llm, tools)

# 그래프 구조 확인
agent.get_graph().print_ascii()

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


/var/folders/n1/c39lcr157bv7z4ynyy6nz_4w0000gn/T/ipykernel_83950/781176319.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


## 4. 단일 질문 테스트

에이전트가 Tool을 적절히 선택하여 사용하는지 확인합니다.

In [4]:
from langchain_core.messages import HumanMessage

# 테스트 1: 계산 질문
print("=== 계산 질문 ===")
result = agent.invoke({
    "messages": [HumanMessage(content="1234 * 5678은 얼마인가요?")]
})
# 전체 메시지 흐름 출력
for msg in result["messages"]:
    if msg.type == "human":
        print(f"[사용자] {msg.content}")
    elif msg.type == "ai" and msg.content:
        print(f"[AI] {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[Tool 호출] {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"[Tool 결과] {msg.content}")

print()

# 테스트 2: 시간 질문
print("=== 시간 질문 ===")
result = agent.invoke({
    "messages": [HumanMessage(content="지금 몇 시야?")]
})
print(f"최종 응답: {result['messages'][-1].content}")
print()

# 테스트 3: 검색 질문
print("=== 검색 질문 ===")
result = agent.invoke({
    "messages": [HumanMessage(content="ReAct 패턴이 뭔지 검색해서 알려줘")]
})
print(f"최종 응답: {result['messages'][-1].content}")

=== 계산 질문 ===
[사용자] 1234 * 5678은 얼마인가요?
[Tool 호출] calculate({'expression': '1234 * 5678'})
[Tool 결과] 1234 * 5678 = 7006652
[AI] 1234 * 5678 = 7,006,652

=== 시간 질문 ===
최종 응답: 지금 14시 09분이야 (오후 2시 09분).

=== 검색 질문 ===
최종 응답: 검색해 봤습니다. 요약하면 ReAct는 LLM(대형 언어 모델)을 ‘사고(reasoning)’와 ‘행동(acting)’을 함께 작동시키는 에이전트 패턴입니다. 핵심 내용과 예시, 장단점을 간단히 정리해 드립니다.

핵심 아이디어
- 모델의 내부 추론(Thought)과 외부 행동(Action)을 반복적으로 섞어 사용합니다.
- 행동(Action)은 툴 호출(웹 검색, 계산기, 데이터베이스 질의 등)이나 구체적 작업 지시를 의미하고, 행동의 결과는 관찰(Observation)으로 들어옵니다.
- 구조적 포맷(예: Thought -> Action -> Observation -> Thought -> ... -> Final Answer)을 통해 다단계 문제 해결과 도구 활용을 자연스럽게 결합합니다.

대표 포맷(예)
- Thought: (모델의 사고/추론)
- Action: (툴 사용 지시, 예: search["query"] / calc["2+2"])
- Observation: (툴/환경이 반환한 결과)
- Thought: ...
- Action: FinalAnswer["최종 답변 내용"]

간단한 예시
- Thought: 이 사람의 생년월일을 확인해야 한다.
- Action: search["홍길동 생년월일"]
- Observation: "홍길동은 1980년 5월 3일 출생"
- Thought: 생년월일로 나이를 계산해야 한다.
- Action: calc["2026-1980"]
- Observation: "46"
- Thought: (최종 정리)
- Action: FinalAns

## 5. 다중 스텝 추론 테스트

연구 보조 도구를 만들어 여러 단계의 추론이 필요한 질문을 테스트합니다.

In [5]:
# 연구용 Tool 추가
@tool
def search_papers(topic: str) -> str:
    """학술 논문을 검색합니다. 연구 주제를 입력하면 관련 논문 목록을 반환합니다."""
    papers = {
        "transformer": "1) Attention Is All You Need (Vaswani et al., 2017)\n2) BERT: Pre-training of Deep Bidirectional Transformers (Devlin et al., 2019)\n3) GPT-4 Technical Report (OpenAI, 2023)",
        "agent": "1) ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)\n2) Toolformer: Language Models Can Teach Themselves to Use Tools (Schick et al., 2023)\n3) AutoGPT: An Autonomous GPT-4 Experiment (2023)",
        "rag": "1) Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (Lewis et al., 2020)\n2) Self-RAG: Learning to Retrieve, Generate, and Critique (Asai et al., 2023)",
    }
    for key, value in papers.items():
        if key in topic.lower():
            return value
    return f"'{topic}' 관련 논문: 다양한 연구가 진행 중입니다."

@tool
def summarize_text(text: str) -> str:
    """텍스트를 요약합니다. 긴 텍스트를 핵심 내용 위주로 요약합니다."""
    # 시뮬레이션 (실제로는 LLM 호출)
    words = text.split()
    if len(words) > 20:
        return "요약: " + " ".join(words[:20]) + "..."
    return f"요약: {text}"

@tool
def compare_items(item1: str, item2: str) -> str:
    """두 항목을 비교 분석합니다."""
    return f"비교 분석: '{item1}' vs '{item2}' - 두 항목 모두 AI 분야에서 중요한 개념이며, 각각 고유한 장점이 있습니다."

# 연구용 에이전트 생성
research_tools = [search_papers, summarize_text, compare_items, calculate]
research_agent = create_react_agent(llm, research_tools)

# 복합 질문 테스트
print("=== 다중 스텝 추론 ===")
result = research_agent.invoke({
    "messages": [HumanMessage(
        content="AI agent 관련 논문을 찾아보고, 어떤 연구들이 있는지 요약해줘"
    )]
})

# 전체 실행 과정 출력
for msg in result["messages"]:
    if msg.type == "human":
        print(f"\n[사용자] {msg.content}")
    elif msg.type == "ai" and msg.content:
        print(f"\n[AI 응답] {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"\n[Tool 호출] {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"[Tool 결과] {msg.content[:100]}..." if len(msg.content) > 100 else f"[Tool 결과] {msg.content}")

=== 다중 스텝 추론 ===


/var/folders/n1/c39lcr157bv7z4ynyy6nz_4w0000gn/T/ipykernel_83950/2986885625.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(llm, research_tools)



[사용자] AI agent 관련 논문을 찾아보고, 어떤 연구들이 있는지 요약해줘

[Tool 호출] search_papers({'topic': 'AI agents (AI 에이전트)'})
[Tool 결과] 1) ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)
2) Toolformer: Language Models Can Tea...

[AI 응답] 아래는 “AI 에이전트(LLM 기반 에이전트)” 관련 주요 논문(검색 결과)과 각 연구의 핵심 아이디어·방법·결과·한계점을 간단히 요약한 내용입니다. 원하시면 각 논문을 더 깊게 파고들어 요약문(방법·실험·수식 등 상세)이나 원문 링크, 후속 연구 목록도 제공해 드립니다.

1) ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)
- 핵심 아이디어: LLM이 문제를 푸는 ‘추론(reasoning)’과 외부 도구/환경와 상호작용하는 ‘행동(acting)’을 분리하지 않고, 추론(생각, chain-of-thought)과 행동(툴 호출, 환경 질의)을 교차(interleave)시켜 수행하도록 설계함.
- 방법: 모델 출력에 “생각(thought)”과 “행동(action)”을 명시적으로 섞어 표현하게 하고, 행동은 환경 쿼리나 API 호출로 구현. 모델은 연속적인 상호작용 속에서 자체적으로 왜 그런 행동을 했는지(추론도) 남김.
- 결과: QA, 추론 기반의 환경 상호작용, multi-step reasoning + tool use에서 성능 향상. 행동의 근거(추론)를 남겨 디버깅·설명 가능성 개선.
- 한계/논의: 행동·추론을 섞는 방식은 유연하지만 여전히 잘못된 행동/환각(hallucination)을 할 수 있음. 학습상 효율성·안정성, 실환경에 적용 시 안전성 문제(무분별한 액세스 등) 존재.

2) Toolformer: Language Models Can Teach Themselves to Use Tools 

## 6. 시스템 프롬프트로 역할 부여

`create_react_agent`에 `prompt` 파라미터로 시스템 프롬프트를 전달할 수 있습니다.

In [6]:
from langchain_core.messages import SystemMessage

# 시스템 프롬프트로 역할 부여
system_prompt = """당신은 친절한 AI 수학 튜터입니다.
학생의 질문에 대해:
1. 먼저 개념을 쉽게 설명하고
2. calculate 도구로 예시 계산을 보여주고
3. 학생이 이해했는지 확인하는 질문을 던지세요
항상 한국어로 답변하세요."""

# 시스템 프롬프트가 포함된 에이전트
math_tutor = create_react_agent(
    llm,
    [calculate],
    prompt=system_prompt
)

# 테스트
result = math_tutor.invoke({
    "messages": [HumanMessage(content="피타고라스 정리가 뭐예요? 예시로 보여주세요.")]
})
print(result["messages"][-1].content)

/var/folders/n1/c39lcr157bv7z4ynyy6nz_4w0000gn/T/ipykernel_83950/3549515514.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_tutor = create_react_agent(


좋아요! 차근차근 설명해드릴게요.

1) 개념(간단히)
- 피타고라스 정리: 직각삼각형에서 두 직각 변(밑변과 높이)의 제곱의 합은 빗변(가장 긴 변)의 제곱과 같다.
- 공식으로: a^2 + b^2 = c^2 (여기서 c는 빗변)

2) 예시 계산 (도구로 계산한 결과 포함)
- 예: 한 직각삼각형의 두 직각 변이 3과 4일 때 빗변을 구해보자.
  - 3^2 = 9  (계산 결과: 9)
  - 4^2 = 16 (계산 결과: 16)
  - 9 + 16 = 25
  - 빗변 c = sqrt(25) = 5  (계산 결과: 5.0)
  → 따라서 3, 4, 5는 피타고라스 정리를 만족하는 한 쌍(3-4-5 삼각형)입니다.

- 반대로: 빗변이 13이고 한 변이 5일 때 다른 한 변을 구하면,
  - 13^2 = 169  (계산 결과: 169)
  - 5^2 = 25    (계산 결과: 25)
  - 다른 변^2 = 169 - 25 = 144
  - 다른 변 = sqrt(144) = 12
  → 그래서 5-12-13도 피타고라스 정리를 만족합니다.

3) 이해 확인 질문
- 연습 문제: 직각삼각형의 두 변이 6과 8일 때 빗변은 얼마일까요? 한 번 계산해 보시겠어요? 답을 적어주시면 같이 확인해드릴게요.


## 7. 대화 히스토리 유지 (ResearchAssistant 클래스)

에이전트가 이전 대화를 기억하도록 히스토리를 관리하는 클래스를 만듭니다.

In [7]:
class ResearchAssistant:
    """대화 히스토리를 유지하는 연구 보조 에이전트"""
    
    def __init__(self):
        self.agent = create_react_agent(
            llm,
            [search_web, search_papers, calculate, get_current_time],
            prompt="당신은 연구 보조 AI입니다. 한국어로 답변하세요. 이전 대화 내용을 참고하여 일관성 있게 답변하세요."
        )
        self.history = []  # 대화 히스토리
    
    def chat(self, user_input: str) -> str:
        """사용자 입력에 대해 응답하고 히스토리를 유지합니다."""
        # 새 사용자 메시지 추가
        self.history.append(HumanMessage(content=user_input))
        
        # 에이전트 실행 (전체 히스토리 전달)
        result = self.agent.invoke({"messages": self.history})
        
        # AI 응답을 히스토리에 추가
        ai_message = result["messages"][-1]
        self.history.append(ai_message)
        
        return ai_message.content
    
    def show_history(self):
        """대화 히스토리를 출력합니다."""
        for msg in self.history:
            role = "사용자" if msg.type == "human" else "AI"
            print(f"[{role}] {msg.content[:200]}")
        print(f"\n총 {len(self.history)}개의 메시지")

# 사용 예시
assistant = ResearchAssistant()

# 첫 번째 대화
print("=== 대화 1 ===")
response = assistant.chat("LangGraph에 대해 검색해줘")
print(response)
print()

# 두 번째 대화 (이전 대화 참조)
print("=== 대화 2 ===")
response = assistant.chat("방금 알려준 내용을 한 줄로 요약해줘")
print(response)
print()

# 히스토리 확인
print("=== 대화 히스토리 ===")
assistant.show_history()

=== 대화 1 ===


/var/folders/n1/c39lcr157bv7z4ynyy6nz_4w0000gn/T/ipykernel_83950/1072405050.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  self.agent = create_react_agent(


간단히 요약해 드리면 다음과 같습니다.

요약
- LangGraph는 에이전트 오케스트레이션(여러 AI 에이전트·작업을 조정)용 프레임워크로 소개됩니다.
- 핵심 아이디어는 '상태 기반(state-based) 그래프'로 복잡한 워크플로우를 노드(상태)와 전이(트리거/액션)로 모델링해 구성·관리하는 것이라는 설명이 있습니다.
- LangChain 계열(또는 LangChain 팀 관련)로 소개되는 경우가 있습니다.

주요 특징(일반적)
- 상태/노드 기반 그래프 모델로 워크플로우를 시각적·구조적으로 정의
- 여러 에이전트(LLM, 도구, 외부 API 등)를 그래프의 다른 노드·액션으로 연결 가능
- 조건부 전이, 병렬 처리, 재시도/에러 핸들링 등 복잡한 흐름 제어 지원
- LangChain 같은 기존 툴과의 통합 또는 상호 보완 사용 가능

대표적 사용 사례
- 복잡한 대화형 에이전트(다단계 의사결정)
- 데이터 파이프라인형 자동화(추출→분석→보고)
- 멀티-모델 협업 워크플로우(검색, 생성, 검증을 분리)

다음으로 도와드릴 수 있는 것
- 공식 문서, GitHub 저장소, 설치 방법(pip/버전)과 예제 코드(간단한 그래프 정의·실행)를 찾아서 링크와 함께 제공
- LangChain과의 차이점/장단점 비교
- 간단한 예제(파이썬 의사코드)로 워크플로우 구성 방법 설명

어떤 정보를 더 원하시나요? (예: GitHub/문서 링크, 설치·예제 코드, LangChain 비교 중 선택)

=== 대화 2 ===
LangGraph는 상태 기반 그래프로 여러 AI 에이전트와 도구를 노드·전이로 모델링해 복잡한 워크플로우를 구성·오케스트레이션하는 프레임워크입니다.

=== 대화 히스토리 ===
[사용자] LangGraph에 대해 검색해줘
[AI] 간단히 요약해 드리면 다음과 같습니다.

요약
- LangGraph는 에이전트 오케스트레이션(여러 AI 에이전트·작업을 조정)용 프레임워크로 소개됩니다.
- 핵심 아이디어는 '상태 기반(state-based) 그래프'로 복잡한 

## 8. 실습 과제: 나만의 ReAct 에이전트 만들기

### 요구사항
1. 자신만의 Tool 3개 이상을 정의하세요 (예: 번역, 코드 실행, 날씨 등)
2. 시스템 프롬프트로 에이전트에 특정 역할을 부여하세요
3. `create_react_agent`로 에이전트를 생성하세요
4. 대화 히스토리를 유지하는 클래스로 감싸세요
5. 최소 3턴의 대화로 테스트하세요

In [9]:
# 8번 실습: 가장 가까운 공휴일 / 선선한 날씨 / 금정구 근처 카페 도구 + ReAct 플래너
from datetime import date, timedelta
import json
import urllib.request
from urllib.parse import urlencode

from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

try:
    import holidays
except ImportError:  # pip install holidays
    holidays = None


# --- 1) Tool 정의 ---
@tool
def get_nearest_korean_public_holiday(reference_date: str = "") -> str:
    """대한민국 기준으로 reference_date(YYYY-MM-DD) 이후 가장 가까운 공휴일 날짜와 이름을 알려줍니다. 빈 문자열이면 오늘 날짜 기준."""
    if holidays is None:
        return "공휴일 계산에 `holidays` 패키지가 필요합니다. 터미널에서: pip install holidays"
    ref = reference_date.strip()[:10]
    if ref:
        try:
            start = date.fromisoformat(ref)
        except ValueError:
            return f"날짜 형식 오류: {reference_date!r}. YYYY-MM-DD 로 넣어 주세요."
    else:
        start = date.today()
    years = range(start.year, start.year + 3)
    kr = holidays.SouthKorea(years=years)
    for i in range(800):
        d = start + timedelta(days=i)
        if d in kr:
            names = kr[d]
            if not isinstance(names, list):
                names = [names]
            label = ", ".join(str(n) for n in names)
            return f"가장 가까운 공휴일은 {d.isoformat()} ({label}) 입니다."
    return "가까운 공휴일을 찾지 못했습니다."


@tool
def get_cool_weather_days_geumjeong(days_ahead: int = 14) -> str:
    """부산 금정구 인근(위도·경도 고정) 일기예보에서 '선선한' 날을 골라 줍니다. 기준: 일 최고기온 18~28°C (Open-Meteo API, 인터넷 필요)."""
    n = max(1, min(int(days_ahead), 16))
    lat, lon = 35.2431, 129.0882
    q = urlencode(
        {
            "latitude": lat,
            "longitude": lon,
            "daily": "temperature_2m_max,temperature_2m_min",
            "timezone": "Asia/Seoul",
            "forecast_days": n,
        }
    )
    url = f"https://api.open-meteo.com/v1/forecast?{q}"
    try:
        with urllib.request.urlopen(url, timeout=12) as resp:
            data = json.loads(resp.read().decode())
    except Exception as e:
        return f"날씨 API 오류: {e}"
    times = data.get("daily", {}).get("time", [])
    tmax = data.get("daily", {}).get("temperature_2m_max", [])
    tmin = data.get("daily", {}).get("temperature_2m_min", [])
    picked = []
    for i, ds in enumerate(times):
        if i >= len(tmax) or i >= len(tmin):
            break
        mx, mn = tmax[i], tmin[i]
        if mx is None or mn is None:
            continue
        if 18.0 <= float(mx) <= 28.0:
            picked.append(f"{ds}: 최고 {mx:.0f}°C / 최저 {mn:.0f}°C")

    head = "부산 금정구 인근 예보 중 선선한 날(최고 18~28°C):\n"
    if not picked:
        return head + "(해당 기간에 기준을 만족하는 날이 없습니다. 기간을 늘려 다시 확인해 보세요.)"
    return head + "\n".join(picked)


@tool
def find_cafes_near_geumjeong_gu(keyword: str = "") -> str:
    """부산 금정구·부산대·장산역·장전역·온천천 근처 카페 후보(데모 목록). keyword에 '뷰','베이커리','브런치' 등을 주면 비슷한 곳 위주로 골라 줍니다."""
    cafes = [
        ("카페 매미사거리", "장전동", "창가 좌석, 커피·케이크, 조용한 편"),
        ("모모스 커피 부산대", "장전동(부산대 정문)", "에스프레소·핸드드립, 학생·방문객 많음"),
        ("어느멋진날", "금정구 두구동", "감성 인테리어·디저트, 사진 찍기 좋음"),
        ("테라로사 장산", "장산동", "넓은 시트, 브런치·베이커리, 가족 단위"),
        ("아날로그 가든", "구서동", "정원 느낌 테라스·산책 후 휴식"),
        ("투썸플레이스 금정본점 인근 독립 카페거리", "서동", "동네 골목 소형 카페 여러 곳"),
        ("온천천 산책로 카페거리", "온천천 인근", "산책→카페 코스 짜기 좋음"),
    ]
    k = (keyword or "").strip().lower()
    lines = []
    for name, area, desc in cafes:
        blob = f"{name} {area} {desc}".lower()
        if not k or k in blob or any(tok in blob for tok in k.split()):
            lines.append(f"- {name} ({area}): {desc}")
    if not lines:
        lines = [f"- {c[0]} ({c[1]}): {c[2]}" for c in cafes]
    return "금정구 근처 카페 후보(데모 데이터, 실제 영업시간은 지도 앱에서 확인):\n" + "\n".join(lines)


outing_tools = [
    get_nearest_korean_public_holiday,
    get_cool_weather_days_geumjeong,
    find_cafes_near_geumjeong_gu,
]

OUTING_SYSTEM_PROMPT = """당신은 부산 금정구 근처에서 '공휴일에 맞춰' 카페 나들이 계획을 짜 주는 플래너입니다.
규칙:
- 가장 가까운 공휴일이 언제인지 알아야 하면 반드시 get_nearest_korean_public_holiday 를 사용합니다(날짜가 정해져 있으면 reference_date에 YYYY-MM-DD).
- 그날이나 그 주의 날씨가 선선한지 보려면 get_cool_weather_days_geumjeong 를 사용합니다.
- 구체적인 카페 후보는 find_cafes_near_geumjeong_gu 로만 제시합니다(직접 지어내지 마세요).
- 도구 결과를 바탕으로 한국어로 친근하게 요약하고 출발 전 확인할 점(영업시간, 예약)을 한두 줄 덧붙입니다."""


class OutingPlannerAgent:
    """공휴일·날씨·카페 도구를 쓰는 ReAct 에이전트 (히스토리 유지)."""

    def __init__(self):
        self.agent = create_react_agent(
            llm,
            outing_tools,
            prompt=OUTING_SYSTEM_PROMPT,
        )
        self.history: list = []

    def chat(self, user_input: str) -> str:
        self.history.append(HumanMessage(content=user_input))
        result = self.agent.invoke({"messages": self.history})
        ai_message = result["messages"][-1]
        self.history.append(ai_message)
        return ai_message.content


planner = OutingPlannerAgent()

# --- 5) 테스트 (3턴 이상) ---
print("=== 1턴: 가까운 공휴일 ===")
print(planner.chat("이번에 부산 금정구 근처에서 놀러 나갈 건데, 오늘 기준으로 가장 가까운 한국 공휴일이 언제야?"))
print()

print("=== 2턴: 그때쯤 선선한 날 ===")
print(
    planner.chat(
        "그 공휴일 전후로 금정구는 날씨가 선선한 날이 언제로 보여? 예보 도구로 알려줘."
    )
)
print()

print("=== 3턴: 카페 추천 ===")
print(
    planner.chat(
        "그럼 그날 산책하기 좋은 분위기에 디저트 맛집 느낌 나는 카페 후보를 골라 줘."
    )
)
print()

print("=== (보너스) 4턴: 키워드 바꿔 재검색 ===")
print(planner.chat("뷰나 테라스 좋은 곳 위주로 다시 찾아줄래?"))


/var/folders/n1/c39lcr157bv7z4ynyy6nz_4w0000gn/T/ipykernel_83950/2888663562.py:124: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  self.agent = create_react_agent(


=== 1턴: 가까운 공휴일 ===
가장 가까운 공휴일은 2026-05-05(어린이날)입니다.

요약: 어린이날에 금정구 근처로 카페 나들이 가기 좋게 도와드릴게요. 원하시면 그날 또는 그 주의 선선한 날(일 최고기온 18~28°C) 확인해 드리고, 금정구 인근 카페 후보도 추천해 드릴게요.

출발 전 확인할 점: 방문 예정 카페의 영업시간/휴무일과 예약 필요 여부를 꼭 확인하세요. 원하시면 날씨 확인이나 카페 후보 추려드릴까요?

=== 2턴: 그때쯤 선선한 날 ===
요청하신 대로 어린이날(2026-05-05) 전후 금정구의 ‘선선한’ 날(일 최고기온 18~28°C) 예보를 확인해봤어요. 다음 날짜들이 해당됩니다:

- 2026-04-10: 최고 22°C / 최저 12°C  
- 2026-04-11: 최고 23°C / 최저 9°C  
- 2026-04-12: 최고 20°C / 최저 10°C  
- 2026-04-13: 최고 20°C / 최저 10°C  
- 2026-04-15: 최고 20°C / 최저 11°C  
- 2026-04-16: 최고 18°C / 최저 12°C  
- 2026-04-18: 최고 21°C / 최저 13°C  
- 2026-04-19: 최고 20°C / 최저 14°C  
- 2026-04-20: 최고 23°C / 최저 15°C  
- 2026-04-21: 최고 23°C / 최저 16°C  
- 2026-04-22: 최고 18°C / 최저 16°C  
- 2026-04-23: 최고 25°C / 최저 14°C  
- 2026-04-25: 최고 20°C / 최저 13°C

요약: 공휴일인 2026-05-05(어린이날) 바로 전후(4월 말~5월 초)에는 이번 예보 범위에서 선선한 날로 분류된 날짜가 나타나지 않았습니다. 가장 가까운 선선한 날은 4월 중순~25일 사이입니다. 공휴일에 꼭 나들이 가고 싶다면 기상 예보를 출발 3일 전 다시 확인하는 걸 권해드립니다.

출발 전 확인할 점: 가려는 카페의 영업시간·휴무와 예약 필요 여부를 꼭 확인